# MPI-Enhanced Disaster Tweet Classification

This notebook implements the **Mathematical Principles of Intelligence (MPI)** theory to improve NLP classification.

## Key Features
1. **SPHA (Softmax-Projected Hyper-Attention)**: Replaces standard attention with e-base scaled attention.
2. **Cognitive Holonomy Regularization**: Adds a loss term to penalize non-smooth metric evolution.

Reference: [Mathematical Principles of Intelligence](https://github.com/simulai/Mathematical-Principles-of-Intelligence)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModel
import os
import math
from tqdm.notebook import tqdm

# Check device
print("MPI Version: 1.1 (DistilBERT + SPHA + CognitiveHolonomy) - Loaded Successfully")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Define Core Theory Modules (SPHA & Cognitive Holonomy)

In [ ]:
class SPHA(nn.Module):
    """
    Softmax-Projected Hyper-Attention (SPHA)
    Implements the e-base scaling law by scaling attention scores by ln(b)/b.
    """
    def __init__(self, embed_dim, num_heads, branching_factor=2.718):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.b = branching_factor
        
        # Scaling factor derived from e-base law: (ln b) / b
        self.mpi_scale = (math.log(self.b) / self.b) * math.sqrt(self.head_dim)
        
        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)

    def forward(self, x, mask=None):
        B, L, D = x.shape
        q = self.q_proj(x).view(B, L, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, L, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, L, self.num_heads, self.head_dim).transpose(1, 2)
        
        # MPI Scaled Dot-Product Attention
        scores = torch.matmul(q, k.transpose(-2, -1)) * self.mpi_scale
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        
        attn_weights = F.softmax(scores, dim=-1)
        out = torch.matmul(attn_weights, v)
        out = out.transpose(1, 2).contiguous().view(B, L, D)
        return self.out_proj(out)

class HolonomyLoss(nn.Module):
    def __init__(self, lambda_h=0.1):
        super().__init__()
        self.lambda_h = lambda_h
        
    def forward(self, hidden_states):
        # Penalize rapid metric changes (Ricci flow smoothing)
        diff = hidden_states[:, 1:, :] - hidden_states[:, :-1, :]
        loss = torch.mean(diff ** 2)
        return self.lambda_h * loss

## 2. Model Architecture

In [ ]:
class MPIDisasterModel(nn.Module):
    def __init__(self, model_name="distilbert-base-uncased", num_classes=2):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        self.hidden_dim = self.backbone.config.hidden_size
        self.mpi_block = SPHA(self.hidden_dim, num_heads=8, branching_factor=math.e)
        self.classifier = nn.Linear(self.hidden_dim, num_classes)
        self.holonomy_loss_fn = HolonomyLoss(lambda_h=0.05)

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden_state = outputs.last_hidden_state
        mpi_out = self.mpi_block(last_hidden_state, attention_mask.unsqueeze(1).unsqueeze(2))
        cls_token = mpi_out[:, 0, :]
        logits = self.classifier(cls_token)
        loss = None
        if labels is not None:
            ce_loss = F.cross_entropy(logits, labels)
            h_loss = self.holonomy_loss_fn(mpi_out)
            loss = ce_loss + h_loss
        return logits, loss

## 3. Data Loading & Training

In [ ]:
class TweetDataset(Dataset):
    def __init__(self, csv_file, tokenizer, max_len=128, is_test=False):
        self.df = pd.read_csv(csv_file)
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.is_test = is_test
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        text = self.df.iloc[idx]['text']
        inputs = self.tokenizer(text, max_length=self.max_len, padding='max_length', truncation=True, return_tensors='pt')
        item = {'input_ids': inputs['input_ids'].squeeze(), 'attention_mask': inputs['attention_mask'].squeeze()}
        if not self.is_test: item['labels'] = torch.tensor(self.df.iloc[idx]['target'], dtype=torch.long)
        return item

def train(model, loader, optimizer):
    model.train()
    total_loss = 0
    loop = tqdm(loader, desc='Training', leave=False)
    for batch in loop:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        _, loss = model(input_ids, mask, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        loop.set_postfix(loss=loss.item())
    return total_loss / len(loader)

# Execution
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
# Check for Kaggle input paths
base_path = "/kaggle/input/nlp-getting-started" if os.path.exists("/kaggle/input/nlp-getting-started") else "data/nlp-getting-started"
train_dataset = TweetDataset(f"{base_path}/train.csv", tokenizer)
test_dataset = TweetDataset(f"{base_path}/test.csv", tokenizer, is_test=True)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

model = MPIDisasterModel().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

print("Starting training...")
for epoch in range(3):
    loss = train(model, train_loader, optimizer)
    print(f"Epoch {epoch+1}, Loss: {loss:.4f}")

## 4. Inference & Submission

In [ ]:
model.eval()
preds = []
with torch.no_grad():
    for batch in tqdm(test_loader, desc='Predicting'):
        input_ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        logits, _ = model(input_ids, mask)
        preds.extend(torch.argmax(logits, dim=1).cpu().numpy())

sub_df = pd.read_csv(f"{base_path}/test.csv")
sub_df['target'] = preds
submission = sub_df[['id', 'target']]
submission.to_csv("submission.csv", index=False)
print("Submission saved!")